In [1]:
# -*- coding: utf-8 -*-
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import DataLoader
import os

# 设置本地模型路径
LOCAL_MODEL_PATH = './bert-base-uncased-local'

# 检查本地模型是否存在
def check_local_model():
    """检查本地模型文件是否存在"""
    required_files = ['config.json', 'pytorch_model.bin', 'vocab.txt', 'tokenizer_config.json']
    
    if not os.path.exists(LOCAL_MODEL_PATH):
        print(f"错误: 本地模型目录 {LOCAL_MODEL_PATH} 不存在")
        print("请先运行 download_model.py 下载模型")
        return False
    
    # 检查必要文件
    existing_files = os.listdir(LOCAL_MODEL_PATH)
    missing_files = [f for f in required_files if f not in existing_files]
    
    if missing_files:
        print(f"错误: 本地模型文件不完整，缺少文件: {missing_files}")
        return False
    
    print(f"找到本地模型: {LOCAL_MODEL_PATH}")
    return True

# 从文件中读取数据
def load_data(file_path):
    texts = []
    labels = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            label, text = line.strip().split('\t')
            labels.append(int(label))
            texts.append(text)
    return texts, labels


# 加载数据
print("加载数据...")
train_texts, train_labels = load_data('train.txt')
print(f"加载了 {len(train_texts)} 条数据")

# 划分训练集和测试集
train_texts, test_texts, train_labels, test_labels = train_test_split(
    train_texts, train_labels, test_size=0.2, random_state=42
)

print(f"训练集: {len(train_texts)} 条，测试集: {len(test_texts)} 条")

# 填空1：加载BERT Tokenization（从本地加载）
print("\n从本地加载BERT Tokenizer...")

# tokenizer = ______(LOCAL_MODEL_PATH)
tokenizer = BertTokenizer.from_pretrained(LOCAL_MODEL_PATH)

# train_encodings = ______(train_texts, ______=True, ______=True, max_length=128)
# test_encodings  = ______(test_texts,  ______=True, ______=True, max_length=128)
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)

print(f"文本编码完成")

# 创建数据集类
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, train_labels)
test_dataset = TextDataset(test_encodings, test_labels)

# 填空2：使用 DataLoader,进行数据加载优化，提高数据加载效率
# train_loader = ______(______, ______=16, ______=True)
# test_loader  = ______(______, ______=16)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

# 模型训练
print("\n从本地加载BERT模型...")
if 'model' not in locals():  # 如果前面没有创建model
    if check_local_model():
        model = BertForSequenceClassification.from_pretrained(
            LOCAL_MODEL_PATH, 
            num_labels=2  # 二分类任务
        )
    else:
        # 如果本地没有，使用之前可能下载的
        model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

print(f"模型加载完成，参数量: {sum(p.numel() for p in model.parameters()):,}")

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",  # 新版参数名
)

#Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

# 训练模型
print("\n开始训练模型...")
trainer.train()  #用训练集对模型微调
print("训练完成!")

# 评估模型
print("\n评估模型...")
predictions = trainer.predict(test_dataset)
predicted_labels = predictions.predictions.argmax(axis=1)

# 输出分类报告
print("\n分类报告:")
print(classification_report(test_labels, predicted_labels, 
                          target_names=['类别0', '类别1']))

# 设置模型为评估模式
model.eval()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

# 测试句子
test_sentence = "I found the Da Vinci Code to be very intriguing and well-written."

test_encoding = tokenizer(
    test_sentence, 
    truncation=True, 
    padding=True, 
    max_length=128,
    return_tensors='pt'
).to(device)

# 进行预测
with torch.no_grad():
    outputs = model(**test_encoding)
    logits = outputs.logits  # 获取 logits

# 获取预测标签
predictions = logits.argmax(dim=1)  # 沿着标签维度取 argmax

# 输出预测标签
predicted_label = predictions.item()  # 取出单个预测结果
print(f"\n测试句子: {test_sentence}")
print(f"预测标签: {predicted_label}")

加载数据...
加载了 7086 条数据
训练集: 5668 条，测试集: 1418 条

从本地加载BERT Tokenizer...
文本编码完成

从本地加载BERT模型...
找到本地模型: ./bert-base-uncased-local
模型加载完成，参数量: 109,483,778
文本编码完成

从本地加载BERT模型...
找到本地模型: ./bert-base-uncased-local
模型加载完成，参数量: 109,483,778

开始训练模型...

开始训练模型...


Epoch,Training Loss,Validation Loss
1,0.066300,0.017911


训练完成!

评估模型...



分类报告:
              precision    recall  f1-score   support

         类别0       0.99      1.00      1.00       626
         类别1       1.00      0.99      1.00       792

    accuracy                           1.00      1418
   macro avg       1.00      1.00      1.00      1418
weighted avg       1.00      1.00      1.00      1418


测试句子: I found the Da Vinci Code to be very intriguing and well-written.
预测标签: 1


In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import os
from sklearn.linear_model import LogisticRegression
from transformers import BertTokenizer, BertModel

# 从文件中读取数据
def load_data(file_path):
    texts = []
    labels = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            label, text = line.strip().split('\t')
            labels.append(int(label))
            texts.append(text)
    return texts, labels

# 检查本地模型是否存在
def check_local_model():
    local_path = './bert-base-uncased-local'
    required_files = ['config.json', 'pytorch_model.bin', 'tokenizer_config.json', 'vocab.txt']
    
    if os.path.exists(local_path):
        missing_files = []
        for req_file in required_files:
            if not os.path.exists(os.path.join(local_path, req_file)):
                missing_files.append(req_file)
        
        if missing_files:
            print(f"本地模型文件不完整，缺少文件: {missing_files}")
            return False
        else:
            print("本地模型文件完整，从本地加载...")
            return True
    else:
        print("本地模型目录不存在")
        return False

# 加载数据
train_texts, train_labels = load_data('train.txt')

# 划分训练集和测试集
train_texts, test_texts, train_labels, test_labels = train_test_split(
    train_texts, train_labels, test_size=0.2, random_state=42
)

# 检查GPU可用性
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

print("\n加载BERT (Base)模型和tokenizer...")
local_model_path = './bert-base-uncased-local'

tokenizer = BertTokenizer.from_pretrained(local_model_path)
model = BertModel.from_pretrained(local_model_path).to(device)

model.eval()  # 设置为评估模式

# 填空1：特征提取函数
def get_bert_embeddings(texts, batch_size=16):
    """使用BERT获取文本嵌入特征"""
    embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        # 对文本进行tokenize
        # encoded_input = ______(batch_texts, ______=True, ______=True, ______=512, return_tensors='pt')
        encoded_input = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors='pt'
        ).to(device)

        # 获取模型输出
        with torch.no_grad():
            outputs = model(**encoded_input)
            batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(batch_embeddings)

    return np.vstack(embeddings)


# 提取特征
print("\n提取训练集特征...")
train_embeddings = get_bert_embeddings(train_texts)  
print(f"训练特征维度: {train_embeddings.shape}")

print("提取测试集特征...")
test_embeddings = get_bert_embeddings(test_texts) 
print(f"测试特征维度: {test_embeddings.shape}")
test_embeddings[:3]

print("\n训练逻辑回归模型...")
lr_model = LogisticRegression(
    random_state=42,
    max_iter=100,
    C=1.0,
    solver='liblinear'
)

# 填空2：训练逻辑回归（LR）分类器 的核心操作
# lr_model.fit(________, ________)
lr_model.fit(train_embeddings, train_labels)

# 模型评估
from sklearn.metrics import classification_report, accuracy_score

print("评估模型性能...")
train_predictions = lr_model.predict(train_embeddings)
test_predictions = lr_model.predict(test_embeddings)

train_accuracy = accuracy_score(train_labels, train_predictions)
test_accuracy = accuracy_score(test_labels, test_predictions)

print(f"\n训练集准确率: {train_accuracy:.4f}")
print(f"测试集准确率: {test_accuracy:.4f}")

print("\n详细分类报告:")
print(classification_report(test_labels, test_predictions, target_names=['1', '0']))

使用设备: cuda

加载BERT (Base)模型和tokenizer...

提取训练集特征...

提取训练集特征...
训练特征维度: (5668, 768)
提取测试集特征...
训练特征维度: (5668, 768)
提取测试集特征...
测试特征维度: (1418, 768)

训练逻辑回归模型...
测试特征维度: (1418, 768)

训练逻辑回归模型...
评估模型性能...

训练集准确率: 0.9981
测试集准确率: 0.9831

详细分类报告:
              precision    recall  f1-score   support

           1       0.98      0.98      0.98       626
           0       0.98      0.99      0.98       792

    accuracy                           0.98      1418
   macro avg       0.98      0.98      0.98      1418
weighted avg       0.98      0.98      0.98      1418

评估模型性能...

训练集准确率: 0.9981
测试集准确率: 0.9831

详细分类报告:
              precision    recall  f1-score   support

           1       0.98      0.98      0.98       626
           0       0.98      0.99      0.98       792

    accuracy                           0.98      1418
   macro avg       0.98      0.98      0.98      1418
weighted avg       0.98      0.98      0.98      1418

